In [ ]:
import os

import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
from scipy import stats

COUNTRIES = [
    ('Ethiopia', 'ethiopia.csv'),
    ('Kenya', 'kenya.csv'),
    ('Sudan', 'sudan.csv'),
    ('Tanzania', 'tanzania.csv'),
    ('Nigeria', 'nigeria.csv'),
]

NUMERIC_COLS = ['T2M', 'T2M_MAX', 'T2M_MIN', 'PRECTOTCORR', 'RH2M', 'WS2M', 'WS2M_MAX']
OUTPUT_DIR = os.path.join('..', 'data')
os.makedirs(OUTPUT_DIR, exist_ok=True)

for country_name, file_name in COUNTRIES:
    print(f"\n===== {country_name} =====")
    path = os.path.join('..', file_name)

    if not os.path.exists(path):
        print(f"File not found: {path}. Skipping {country_name}.")
        continue

    df = pd.read_csv(path)
    df['Country'] = country_name
    df['Date'] = pd.to_datetime(df['YEAR'] * 1000 + df['DOY'], format='%Y%j')
    df['Month'] = df['Date'].dt.month

    df.replace(-999, np.nan, inplace=True)

    print("Data loaded. First 5 rows:")
    display(df.head())

    z_scores = np.abs(stats.zscore(df[NUMERIC_COLS], nan_policy='omit'))
    outlier_count = int((z_scores > 3).any(axis=1).sum())
    print(f"Total rows with outliers (|Z| > 3): {outlier_count}")

    df[NUMERIC_COLS] = df[NUMERIC_COLS].ffill()

    monthly_df = df.set_index('Date').resample('M').mean()

    plt.figure(figsize=(14, 6))
    plt.plot(monthly_df.index, monthly_df['T2M'], color='crimson', marker='.')
    warmest_date = monthly_df['T2M'].idxmax()
    warmest_val = monthly_df['T2M'].max()
    coolest_date = monthly_df['T2M'].idxmin()
    coolest_val = monthly_df['T2M'].min()

    plt.annotate(
        f'Warmest ({warmest_val:.1f}°C)',
        xy=(warmest_date, warmest_val),
        xytext=(10, 10),
        textcoords='offset points',
        arrowprops=dict(arrowstyle='->'),
    )
    plt.annotate(
        f'Coolest ({coolest_val:.1f}°C)',
        xy=(coolest_date, coolest_val),
        xytext=(10, -20),
        textcoords='offset points',
        arrowprops=dict(arrowstyle='->'),
    )

    plt.title(f'Monthly Mean Air Temperature ({country_name})')
    plt.grid(True, alpha=0.3)
    plt.show()

    monthly_rain = df.set_index('Date').resample('M')['PRECTOTCORR'].sum()
    plt.figure(figsize=(14, 6))
    monthly_rain.plot(kind='bar', color='skyblue')
    plt.title(f'Monthly Total Precipitation ({country_name})')
    plt.xticks(rotation=45)
    plt.tight_layout()
    plt.show()

    output_file = os.path.join(OUTPUT_DIR, f"{country_name.lower()}_clean.csv")
    df.to_csv(output_file, index=False)
    print(f"Cleaned data saved to {output_file}")